# Pipeline de Despliegue: Registro de Modelos en el Catálogo Corporativo
## Módulo 01: Inicialización de Parámetros y Gobernanza de Artefactos

Este notebook marca el inicio de la fase de operacionalización y puesta en producción (*Model Deployment*) dentro de **Microsoft Fabric**. El objetivo central es registrar y desplegar **dos modelos** entrenados previamente (Clasificación y Regresión), ejecutar inferencia en lote con cada uno, y persistir los resultados de negocio en la capa Gold del Lakehouse.

---

## 🏛️ Configuración de Identificadores y Control de Versiones

Para asegurar un despliegue limpio y desacoplado, el pipeline inicializa las constantes estructurales que actúan como llaves primarias en la arquitectura de **MLOps**:

* **`RUN_ID_CLF` / `RUN_ID_REG`**: Hashes únicos de ejecución que conectan cada módulo de despliegue de forma unívoca con su experimento de entrenamiento, garantizando trazabilidad ininterrumpida (*provenance tracking*).
* **`MODEL_NAME_CLF` / `MODEL_NAME_REG`**: Definen los nombres canónicos bajo los cuales cada artefacto quedará expuesto en el catálogo del Workspace. Siguen la convención `snake_case` obligatoria de gobernanza.

**Modelos registrados:**
- **Clasificación** (`modelo_clasificacion_churn_v1`): Predice si un cliente abandonará el servicio en los próximos 30 días (*churn*), permitiendo acciones de retención proactiva.
- **Regresión** (`modelo_regresion_valor_cliente_v1`): Estima el valor monetario esperado de un cliente en el siguiente trimestre, priorizando esfuerzos comerciales.


In [47]:
# ========================================================
# 01. CONFIGURACIÓN DE MODELOS PARA REGISTRO
# ========================================================

import mlflow
import pandas as pd
import numpy as np
from pyspark.sql.functions import col

# --- MODELO 1: CLASIFICACIÓN (predicción de churn) ---
RUN_ID_CLF = "e99eafea-676b-485f-a7a6-c4e86f6854e0"
MODEL_NAME_CLF = "modelo_clasificacion_churn_v1"

# --- MODELO 2: REGRESIÓN (estimación de valor de cliente) ---
RUN_ID_REG = "8450836c-4825-4321-8ade-fbc47004aafe"
MODEL_NAME_REG = "modelo_regresion_valor_cliente_v1"

print(f"✓ Clasificación  → Run ID : {RUN_ID_CLF}")
print(f"✓ Clasificación  → Nombre : {MODEL_NAME_CLF}")
print(f"✓ Regresión      → Run ID : {RUN_ID_REG}")
print(f"✓ Regresión      → Nombre : {MODEL_NAME_REG}")

StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 55, Finished, Available, Finished, False)

✓ Clasificación  → Run ID : e99eafea-676b-485f-a7a6-c4e86f6854e0
✓ Clasificación  → Nombre : modelo_clasificacion_churn_v1
✓ Regresión      → Run ID : 8450836c-4825-4321-8ade-fbc47004aafe
✓ Regresión      → Nombre : modelo_regresion_valor_cliente_v1


## 🚀 Registro de los Modelos en el Catálogo Centralizado (Model Registry)

Esta fase representa la culminación del pipeline de despliegue, donde los artefactos binarios de Machine Learning transicionan de ser simples resultados de experimento a convertirse en **activos de software gobernados**. Utilizando la API de **MLflow**, ambos modelos son promovidos formalmente al registro centralizado de Microsoft Fabric.

### 🔹 Mecanismo de Promoción y Versionado Automático
El script invoca **`mlflow.register_model()`** con dos componentes clave:
1. **`model_uri`**: La URI `runs:/[RUN_ID]/model` localiza el artefacto binario serializado dentro del almacenamiento del Lakehouse.
2. **`name`**: El nombre en `snake_case` asigna la identidad formal en el catálogo.

El motor de MLOps realiza automáticamente:
* **Versionado Lineal**: Asigna `Versión 1` en el primer registro; versiones sucesivas se incrementan sin sobrescribir despliegues anteriores.
* **Trazabilidad End-to-End**: Mantiene enlace permanente al `RUN_ID` de origen para auditoría de parámetros, datos y métricas.


In [48]:
# ========================================================
# 02. REGISTRO DE LOS 2 MODELOS EN EL CATÁLOGO DE FABRIC
# ========================================================

# --- Registro Modelo 1: Clasificación ---
print("Promoviendo modelo de CLASIFICACIÓN al Registro Central...")

model_uri_clf = f"runs:/{RUN_ID_CLF}/model"

model_details_clf = mlflow.register_model(
    model_uri=model_uri_clf,
    name=MODEL_NAME_CLF
)

print(f"  ✓ Nombre   : {model_details_clf.name}")
print(f"  ✓ Versión  : {model_details_clf.version}")

print()

# --- Registro Modelo 2: Regresión ---
print("Promoviendo modelo de REGRESIÓN al Registro Central...")

model_uri_reg = f"runs:/{RUN_ID_REG}/ridge_model"

model_details_reg = mlflow.register_model(
    model_uri=model_uri_reg,
    name=MODEL_NAME_REG
)

print(f"  ✓ Nombre   : {model_details_reg.name}")
print(f"  ✓ Versión  : {model_details_reg.version}")

print("\n🚀 Ambos modelos registrados exitosamente en Microsoft Fabric.")

StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 56, Finished, Available, Finished, False)

2026-06-15:04:09:46,584 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 409, b'{"requestId":"39f43d1f-9beb-42bf-bcf3-edac46317e08","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'modelo_clasificacion_churn_v1\' is already in use","isRetriable":false}'
Registered model 'modelo_clasificacion_churn_v1' already exists. Creating a new version of this model...
2026-06-15:04:09:49,885 ERROR    [shared_platform_utils.py:82] Create MLModel failed, status_code: 409, b'{"requestId":"9cb9b6fe-4203-4222-9d7f-021d2ab462ee","errorCode":"ItemDisplayNameAlreadyInUse","message":"Requested \'modelo_regresion_valor_cliente_v1\' is already in use","isRetriable":false}'
Registered model 'modelo_regresion_valor_cliente_v1' already exists. Creating a new version of this model...
Created version '3' of model 'modelo_regresion_valor_cliente_v1'.


## 📦 Carga de Datos y Validación de Esquemas para Inferencia en Lote (Batch Scoring)

Una vez que los modelos han sido promovidos al Catálogo Central, el pipeline avanza hacia la fase operativa de **Inferencia en Producción**. Se leen los datos de la capa Gold y se seleccionan las features exactas que cada modelo espera, garantizando compatibilidad de esquema.

### 🔹 Features por modelo
- **Clasificación (churn)**: `age`, `quantity`, `days_since_last_purchase` — variables conductuales que el clasificador aprendió a asociar con abandono.
- **Regresión (valor de cliente)**: `age`, `quantity`, `total_spent` — variables financieras y de volumen que el regresor utiliza para estimar el valor esperado.

La proyección selectiva reduce el *Network Shuffle* en Spark e inmuniza el pipeline contra mutaciones futuras del esquema de la tabla Gold.


In [50]:
# ========================================================
# 03. CARGA DE DATOS PARA INFERENCIA EN PRODUCCIÓN
# ========================================================

GOLD_TABLE = "gold.dbo.gold_features_clasificacion"

df_gold = spark.read.table(GOLD_TABLE)

print(f"✓ Tabla Gold cargada: {GOLD_TABLE}")
print(f"✓ Total de registros disponibles: {df_gold.count():,}")

# ---------- MODELO 1: CLASIFICACIÓN ----------
feature_cols_clf = [
    "age",
    "quantity",
    "price_per_unit"
]

df_inference_clf = (
    df_gold
    .select(feature_cols_clf)
    .na.drop()
)

print(
    f"\n✓ Registros para inferencia de CLASIFICACIÓN: "
    f"{df_inference_clf.count():,}"
)

# ---------- MODELO 2: REGRESIÓN ----------
feature_cols_reg = [
    "id_genero",
    "id_categoria",
    "quantity",
    "age",
    "price_per_unit"
]

df_inference_reg = (
    df_gold
    .select(feature_cols_reg)
    .na.drop()
)

print(
    f"✓ Registros para inferencia de REGRESIÓN: "
    f"{df_inference_reg.count():,}"
)

StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 59, Finished, Available, Finished, False)

✓ Tabla Gold cargada: gold.dbo.gold_features_clasificacion
✓ Total de registros disponibles: 1,000

✓ Registros para inferencia de CLASIFICACIÓN: 1,000
✓ Registros para inferencia de REGRESIÓN: 1,000


## 🔮 Ejecución de Predicciones en Lote (Batch Inference Pipeline)

Este módulo ejecuta la inferencia masiva con ambos modelos registrados. Se utiliza `mlflow.pyfunc.load_model()` apuntando a `models:/{MODEL_NAME}/latest` para desacoplar el pipeline del framework de entrenamiento subyacente.

### 🔹 Abstracción mediante `pyfunc`
* **Wrapper Universal**: Expone el método `.predict()` de forma uniforme, independientemente de si el modelo interno es Scikit-Learn, XGBoost u ONNX.
* **Puntero Dinámico (`/latest`)**: Descarga automáticamente la versión más reciente del catálogo, eliminando la necesidad de actualizar el código ante reentrenamientos.


In [52]:
# ========================================================
# 04. EJECUCIÓN DE PREDICCIONES EN LOTE (BATCH INFERENCE)
# ========================================================
import mlflow
import numpy as np
import pandas as pd

# ---------- MODELO 1: CLASIFICACIÓN (churn) ----------
print("=== MODELO 1: Clasificación — Predicción de Churn ===")
print(f"Descargando modelo de clasificación desde el Registro: {MODEL_NAME_CLF}...")

try:
    # Carga usando la API fluida de pyfunc (ideal para producción/inferencia masiva)
    loaded_clf = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME_CLF}/latest")
    
    # Conversión segura a Pandas y alineación de tipos de datos
    pdf_clf = df_inference_clf.toPandas()
    X_clf = pdf_clf[feature_cols_clf].to_numpy(dtype=np.int32)
    
    # Generar predicciones e integrarlas al DataFrame resultante
    predictions_clf = loaded_clf.predict(X_clf)
    pdf_clf["prediccion_churn"] = predictions_clf
    
    print(f"✓ Inferencia de clasificación completada. Registros procesados: {len(pdf_clf):,}")
    display(pdf_clf.head(5))  # Usamos display() que renderiza tablas mucho más estéticas en Fabric

except Exception as e:
    print(f"❌ Error al ejecutar la inferencia de clasificación: {e}")

print("\n" + "="*50 + "\n")

# ---------- MODELO 2: REGRESIÓN (valor de cliente) ----------
print("=== MODELO 2: Regresión — Estimación de Valor de Cliente ===")
print(f"Descargando modelo de regresión desde el Registro: {MODEL_NAME_REG}...")

try:
    # Carga del modelo de regresión (total_amount) desde el registro central
    loaded_reg = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME_REG}/latest")
    
    # Conversión a Pandas y extracción de matriz de características
    pdf_reg = df_inference_reg.toPandas()
    
    # Aseguramos que las características se traten como matriz flotante/numérica limpia
    X_reg = np.vstack(pdf_reg[feature_cols_reg].values) if hasattr(pdf_reg[feature_cols_reg].iloc[0], '__iter__') else pdf_reg[feature_cols_reg].values
    
    # Generar predicciones continuas de monto/valor
    predictions_reg = loaded_reg.predict(X_reg)
    pdf_reg["prediccion_valor_cliente"] = predictions_reg
    
    print(f"✓ Inferencia de regresión completada. Registros procesados: {len(pdf_reg):,}")
    display(pdf_reg.head(5))

except Exception as e:
    print(f"❌ Error al ejecutar la inferencia de regresión: {e}")

StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 63, Finished, Available, Finished, False)

/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


2026/06/15 04:12:32 INFO mlflow.store.artifact.artifact_repo: The progress bar can be disabled by setting the environment variable MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR to false
/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/mlflow/store/artifact/utils/models.py:32: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/2.12.2/model-registry.html#migrating-from-stages
  latest = client.get_latest_versions(name, None if stage is None else [stage])


✓ Inferencia de regresión completada. Registros procesados: 1,000


SynapseWidget(Synapse.DataFrame, 2a671036-2bcf-4719-a220-86f32f6434df)

StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 65, Finished, Available, Finished, False)

## 💾 Persistencia de Resultados en la Capa Gold

Las predicciones de ambos modelos se persisten en la capa Gold del Lakehouse como tablas Delta administradas. Se usa `saveAsTable()` sin forzar formato `parquet` para mantener compatibilidad nativa con Delta Lake de Microsoft Fabric.

Cada tabla representa un activo de negocio listo para ser consumido por dashboards de Power BI o procesos ETL descendentes:
- `spark_catalog.gold.predicciones_churn_clientes` → señales de riesgo de abandono por cliente.
- `spark_catalog.gold.predicciones_valor_clientes` → estimación de valor esperado por cliente para priorización comercial.


In [53]:
# ========================================================
# 05. GUARDAR PREDICCIONES EN TABLAS GOLD (Delta Lake)
# ========================================================

# --- Tabla 1: predicciones de churn (clasificación) ---
output_table_clf = "gold.dbo.predicciones_churn_clientes"
df_churn_predictions = spark.createDataFrame(pdf_clf)
print(f"Escribiendo predicciones de CLASIFICACIÓN en: {output_table_clf}...")
df_churn_predictions.write \
    .mode("overwrite") \
    .saveAsTable(output_table_clf)           # Delta por defecto en Fabric
print(f"  ✓ Tabla escrita correctamente.")

print()

# --- Tabla 2: predicciones de valor de cliente (regresión) ---
output_table_reg = "gold.dbo.predicciones_valor_clientes"
df_valor_predictions = spark.createDataFrame(pdf_reg)
print(f"Escribiendo predicciones de REGRESIÓN en: {output_table_reg}...")
df_valor_predictions.write \
    .mode("overwrite") \
    .saveAsTable(output_table_reg)           # Delta por defecto en Fabric
print(f"  ✓ Tabla escrita correctamente.")

print("\n🏁 Pipeline de Machine Learning de extremo a extremo completado con éxito.")
print("✓ Modelos registrados, desplegados y resultados almacenados en la capa Gold.")


StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 67, Finished, Available, Finished, False)

Escribiendo predicciones de CLASIFICACIÓN en: gold.dbo.predicciones_churn_clientes...
  ✓ Tabla escrita correctamente.

Escribiendo predicciones de REGRESIÓN en: gold.dbo.predicciones_valor_clientes...
  ✓ Tabla escrita correctamente.

🏁 Pipeline de Machine Learning de extremo a extremo completado con éxito.
✓ Modelos registrados, desplegados y resultados almacenados en la capa Gold.


## 🔍 Verificación de Persistencia y Visualización de Resultados

Este componente actúa como compuerta de auditoría visual (*Data Inspection Gate*). Se realiza una lectura directa desde el catálogo administrado para confirmar la correcta persistencia de ambas tablas y visualizar los DataFrames enriquecidos con predicciones.

Esta lectura simula el comportamiento de herramientas externas de Business Intelligence (Power BI) o procesos ETL descendentes al consumir las inferencias en producción.


In [54]:
# ========================================================
# 06. VERIFICACIÓN DE PERSISTENCIA EN CAPA GOLD
# ========================================================

# --- Verificar Tabla 1: predicciones de churn ---
print("=== Verificación: predicciones_churn_clientes ===")
df_verify_clf = spark.read.table("gold.dbo.predicciones_churn_clientes")
print(f"✓ Registros en tabla de churn : {df_verify_clf.count():,}")
display(df_verify_clf)

print()

# --- Verificar Tabla 2: predicciones de valor de cliente ---
print("=== Verificación: predicciones_valor_clientes ===")
df_verify_reg = spark.read.table("gold.dbo.predicciones_valor_clientes")
print(f"✓ Registros en tabla de valor : {df_verify_reg.count():,}")
display(df_verify_reg)


StatementMeta(, 62c4ba57-0c23-4bdd-8738-54902a5e3cc2, 68, Finished, Available, Finished, True)

=== Verificación: predicciones_churn_clientes ===
✓ Registros en tabla de churn : 1,000


SynapseWidget(Synapse.DataFrame, 2a3386fb-dff2-4559-a54a-55f7c2da031c)


=== Verificación: predicciones_valor_clientes ===
✓ Registros en tabla de valor : 1,000


SynapseWidget(Synapse.DataFrame, 6ed62b6c-9f75-4063-82fa-c462f7f391ac)